# Model Evaluation — companion notebook

Sweep the decision threshold, draw the precision-recall curve, and pick the operating point from the **cost of each error** rather than by feel.

## 1. An imbalanced problem

1.5% positives, and a model that outputs a score rather than a label — which is what a real classifier gives you.

In [1]:
import random
random.seed(11)

N, RATE = 4000, 0.015
truth = [1 if random.random() < RATE else 0 for _ in range(N)]
POS = sum(truth)

rng = random.Random(5)
# scores overlap: positives skew high, negatives skew low, but not cleanly
score = [min(1.0, max(0.0, rng.gauss(0.65, 0.18) if t else rng.gauss(0.30, 0.18)))
         for t in truth]

def confusion(th):
    tp = sum(1 for t, s in zip(truth, score) if t == 1 and s >= th)
    fp = sum(1 for t, s in zip(truth, score) if t == 0 and s >= th)
    fn = sum(1 for t, s in zip(truth, score) if t == 1 and s < th)
    tn = sum(1 for t, s in zip(truth, score) if t == 0 and s < th)
    return tp, fp, fn, tn

def prf(th):
    tp, fp, fn, tn = confusion(th)
    p = tp/(tp+fp) if tp+fp else 0.0
    r = tp/(tp+fn) if tp+fn else 0.0
    f = 2*p*r/(p+r) if p+r else 0.0
    return p, r, f, (tp+tn)/N

print(f"{N} cases, {POS} positive ({POS/N:.1%})")
print(f"always-negative accuracy: {1-POS/N:.1%}  (recall 0.0)")

4000 cases, 44 positive (1.1%)
always-negative accuracy: 98.9%  (recall 0.0)


## 2. The precision-recall curve

Every point is a threshold. There is no point that avoids both errors — the score distributions overlap.

In [2]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

ths = [i/200 for i in range(1, 200)]
P = [prf(t)[0] for t in ths]
R = [prf(t)[1] for t in ths]
F = [prf(t)[2] for t in ths]

fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4))
a1.plot(R, P); a1.set_xlabel("recall"); a1.set_ylabel("precision")
a1.axhline(POS/N, ls="--", lw=.8, color="grey")
a1.annotate("random baseline = base rate", (0.5, POS/N),
            textcoords="offset points", xytext=(0, 8), fontsize=8, color="grey")
a1.set_title("Precision-recall"); a1.grid(alpha=.3)

a2.plot(ths, P, label="precision"); a2.plot(ths, R, label="recall")
a2.plot(ths, F, label="F1", ls="--")
a2.set_xlabel("threshold"); a2.legend(); a2.grid(alpha=.3)
a2.set_title("Every threshold is a trade")
plt.tight_layout(); plt.show()

best_f1 = max(ths, key=lambda t: prf(t)[2])
p, r, f, acc = prf(best_f1)
print(f"best-F1 threshold {best_f1:.2f}: precision {p:.1%} recall {r:.1%} "
      f"F1 {f:.3f} accuracy {acc:.1%}")

best-F1 threshold 0.72: precision 37.3% recall 43.2% F1 0.400 accuracy 98.6%


## 3. Choose the threshold from COST, not from F1

F1 silently assumes the two errors cost the same. They almost never do. Put real numbers on a miss and a false alarm and the optimum moves.

In [3]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

def total_cost(th, cost_fn, cost_fp):
    _tp, fp, fn, _tn = confusion(th)
    return fn*cost_fn + fp*cost_fp

fig, ax = plt.subplots(figsize=(8, 4))
print(f"  {'miss:false-alarm':>18} {'best threshold':>15} {'recall':>8} {'precision':>10}")
for cfn, cfp, label in [(1, 1, "1:1"), (10, 1, "10:1"), (100, 1, "100:1"), (1, 5, "1:5")]:
    costs = [total_cost(t, cfn, cfp) for t in ths]
    ax.plot(ths, [c/max(costs) for c in costs], label=f"miss:alarm = {label}")
    best = min(ths, key=lambda t: total_cost(t, cfn, cfp))
    p, r, _f, _a = prf(best)
    print(f"  {label:>18} {best:>15.2f} {r:>8.1%} {p:>10.1%}")
ax.set_xlabel("threshold"); ax.set_ylabel("total cost (normalised)")
ax.legend(); ax.grid(alpha=.3)
ax.set_title("The optimum moves with the cost ratio -- F1 cannot know this")
plt.tight_layout(); plt.show()
print("\nSame model, same data. Only the price of being wrong changed.")

    miss:false-alarm  best threshold   recall  precision


                 1:1            0.81    22.7%      66.7%


                10:1            0.67    59.1%      24.3%


               100:1            0.47    88.6%       5.8%


                 1:5            0.97     2.3%     100.0%

Same model, same data. Only the price of being wrong changed.


## 4. How much can you trust these numbers?

With few positives, every metric has wide error bars. Bootstrap to see them before claiming one model beats another.

In [4]:
import statistics
idx = list(range(N))
boot = []
for b in range(300):
    rb = random.Random(1000+b)
    sample = [rb.choice(idx) for _ in range(N)]
    t2 = [truth[i] for i in sample]; s2 = [score[i] for i in sample]
    tp = sum(1 for t, s in zip(t2, s2) if t == 1 and s >= 0.5)
    fn = sum(1 for t, s in zip(t2, s2) if t == 1 and s < 0.5)
    boot.append(tp/(tp+fn) if tp+fn else 0.0)
boot.sort()
lo, hi = boot[int(.025*len(boot))], boot[int(.975*len(boot))]
print(f"recall at threshold 0.50: {prf(0.5)[1]:.1%}")
print(f"95% bootstrap interval:   {lo:.1%} to {hi:.1%}   (width {hi-lo:.1%})")
print(f"\nWith only {POS} positives, a difference smaller than that width")
print("is not evidence of anything.")

recall at threshold 0.50: 84.1%
95% bootstrap interval:   71.8% to 94.4%   (width 22.6%)

With only 44 positives, a difference smaller than that width
is not evidence of anything.


## 5. Your turn

1. Replace `score` and `truth` with your own model's outputs.
2. Put your real cost of a miss and of a false alarm into §3 and read the threshold off the curve.
3. Check the §4 interval before reporting that one model beat another.